# 03 — Train local AMR-SAGE (Colab driver)

Phase-1: one hospital → core KG → local GNN, macro-F1 vs the majority baseline.

**Run cells top to bottom** (Runtime → Run all). Only edit the `ARMD_DIR` path in the setup cell.
Per repo convention this notebook only *imports + calls* `src/amr_fed/` modules — no logic here.

In [ ]:
# 1) Get the code (clone the branch, or pull if already cloned)
!git clone -b phase1-core-pipeline https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout phase1-core-pipeline && git pull)
!pip install -q torch_geometric

In [ ]:
# 2) Point at the data. Mount Drive and set ARMD_DIR *before* importing amr_fed.
from google.colab import drive
drive.mount('/content/drive')

import os
# EDIT THIS to your ARMD folder. If ARMD is under 'Shared with me', first add a
# shortcut to it in My Drive, then it appears under /content/drive/MyDrive/.
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'

In [ ]:
# 3) Sanity: config resolves the data dir and sees the CSVs
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix the path in cell 2 and re-run.'
print('CSVs found:', sum((D / f).exists() for f in config.ARMD_TABLES.values()), 'of', len(config.ARMD_TABLES))

In [ ]:
# 4) Smoke test on one small hospital (ICU) — fast end-to-end check
from amr_fed.train_local import main
model, metrics = main(ward='ICU')
print(metrics)

In [ ]:
# 5) Full cohort (all wards) — run once the smoke test prints a macro-F1
from amr_fed.train_local import main
model, metrics = main(ward=None)
print(metrics)

In [ ]:
# 6) ENRICHMENT edge #1: add (patient, has, comorbidity). Compare macro-F1 vs core (0.663).
# First run STREAMS the ~18GB comorbidity CSV once (slow — several minutes from Drive)
# and caches the patient->comorbidity edge list to Drive; later runs reuse the cache.
from amr_fed.train_local import main
cache = '/content/drive/MyDrive/amr_comorbidity_edges.parquet'
model, metrics = main(ward=None, enrich=('comorbidity',), comorbidity_cache=cache)
print(metrics)

In [ ]:
# 7) ENRICHMENT edge #2: add (patient, prior_exposure, antibiotic). Compare vs core (0.663).
# Reuses the antibiotic node (no new node type). Reads abx_class_exposure (~540MB, fast);
# caches the patient->antibiotic exposure edges to Drive for later reuse.
cache = '/content/drive/MyDrive/amr_prior_exposure_edges.parquet'
model, metrics = main(ward=None, enrich=('prior_exposure',), exposure_cache=cache)
print(metrics)

In [ ]:
# 8) RICHER PATIENT FEATURES: add labs/vitals (median values + measured flags) as
# patient NODE features. Not edges — this feeds the decoder directly per patient.
# Compare macro-F1 vs core (0.663). Caches the per-culture labs/vitals summary to Drive.
from amr_fed.train_local import main
cache = '/content/drive/MyDrive/amr_labvital_per_culture.parquet'
model, metrics = main(ward=None, rich_patient=True, labvital_cache=cache)
print(metrics)

In [ ]:
# 9) TUNING PASS: is ~0.66 a real ceiling or just an under-tuned model?
# Build the graph ONCE (with the richest features), then try ~5 architectures:
# sum vs mean aggregation, wider/deeper, lower lr, more regularization.
from amr_fed.graph_build import build_graph
from amr_fed.train_local import sweep
data = build_graph(ward=None, rich_patient=True,
                   labvital_cache='/content/drive/MyDrive/amr_labvital_per_culture.parquet')
results = sweep(data)   # prints a ranked summary; ~10 min total on GPU

In [ ]:
# 10) FULL GRID: every feature combination x architecture sweep (the exhaustive run).
# 16 feature combos (comorbidity/prior_exposure/procedure on-off x labs/vitals on-off)
# x 3 architectures = 48 trainings. ~1 hour on GPU. Prints a running best so partial
# results survive a disconnect, then a Top-10 table.
# Run cells 6, 7, 8 at least once first so the enrichment caches exist (else the first
# comorbidity combo streams the 18GB CSV before caching it).
from amr_fed.train_local import full_grid
results = full_grid(cache_dir='/content/drive/MyDrive', ward=None)
# Want the *complete* sweep (5 archs x 16 = 80 runs, ~2h)? pass the bigger grid:
#   from amr_fed.train_local import _SWEEP_GRID
#   results = full_grid(cache_dir='/content/drive/MyDrive', arch_configs=_SWEEP_GRID)

In [ ]:
# 11) PATIENT-HISTORY PREDICTORS (personalized-antibiogram signal; Corbin 2022).
# Adds per-test prior-resistance features to the GNN decoder: patient's past
# resistance rate (overall / same-antibiotic / same-organism), prior-culture count,
# and recency. All leakage-safe (only cultures BEFORE each test). Also reports AUROC
# (to compare vs the 0.74-0.81 literature) and a val-tuned decision threshold.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.train_local import main
model, metrics = main(ward=None, patient_history=True)
print(metrics)

In [ ]:
# 12) BEST FEATURE SET: patient-history + specimen source (urine/resp/blood).
# Specimen resistance varies a lot by source (EDA: urine ~0.18 vs resp ~0.29) and the
# literature uses it. Adds a 3-way one-hot to the decoder alongside patient-history.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.train_local import main
model, metrics = main(ward=None, patient_history=True, specimen=True)
print(metrics)

In [ ]:
# 13) + PRIOR PRESCRIPTIONS (literature's #2 predictor; Corbin 2022): patient-history +
# prior-antibiotic-exposure features (count / #classes / recency / has-prior). Compare vs
# cell 11 (history only = 0.71 / AUROC 0.84). First run caches the per-culture prescription
# summary to Drive.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.train_local import main
cache = '/content/drive/MyDrive/amr_prescription_history.parquet'
model, metrics = main(ward=None, patient_history=True, prescriptions=True, prescription_cache=cache)
print(metrics)